# Census imports: simple version


## Step 1. Import the libraries

In [1]:
import json
import urllib.request, urllib.parse, urllib.error
import pandas as pd

## Step 2. Settings
Paste your Census API key between the quotes.

In [2]:
API_KEY = __import__("os").environ["CENSUS_API_KEY"]  # set in terminal, never typed here
BASE_URL = "https://api.census.gov/data/timeseries/intltrade/imports/hs"

## Step 3. Build the request
One month (Oct 2023), one country (China = 5700), all HS4 products, all customs districts.

params = {
    "get": "I_COMMODITY,CTY_CODE,DISTRICT,CNT_CHA_MO,CNT_WGT_MO,CNT_VAL_MO,GEN_VAL_MO",
    "time": "2023-10",
    "COMM_LVL": "HS4",
    "CTY_CODE": "5700",
    "I_COMMODITY": "*",
    "DISTRICT": "*",
    "key": API_KEY,
}
url = BASE_URL + "?" + urllib.parse.urlencode(params, safe=",*")
print(url.replace(API_KEY, "YOUR_KEY"))

In [5]:
params = {
    "get": "I_COMMODITY,CTY_CODE,CNT_CHA_MO,CNT_WGT_MO,CNT_VAL_MO,GEN_VAL_MO",
    "time": "2023-10",
    "COMM_LVL": "HS4",
    "CTY_CODE": "5700",
    "I_COMMODITY": "8517",
    "key": API_KEY,
}
url = BASE_URL + "?" + urllib.parse.urlencode(params, safe=",*")
print(url.replace(API_KEY, "YOUR_KEY"))

https://api.census.gov/data/timeseries/intltrade/imports/hs?get=I_COMMODITY,CTY_CODE,CNT_CHA_MO,CNT_WGT_MO,CNT_VAL_MO,GEN_VAL_MO&time=2023-10&COMM_LVL=HS4&CTY_CODE=5700&I_COMMODITY=8517&key=YOUR_KEY


## Step 4. Fetch the data

In [6]:
try:
    with urllib.request.urlopen(url, timeout=120) as response:
        text = response.read().decode()
    print("Fetched", len(text), "characters")
except urllib.error.HTTPError as e:
    text = ""
    print("Census error", e.code, ":", e.read().decode())

Fetched 215 characters


## Step 5. Load into a DataFrame

In [7]:
rows = json.loads(text)                        # list of lists, first row is the header
df = pd.DataFrame(rows[1:], columns=rows[0])
df = df.loc[:, ~df.columns.duplicated()]       # Census sometimes repeats a column

for col in ["CNT_CHA_MO", "CNT_WGT_MO", "CNT_VAL_MO", "GEN_VAL_MO"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.shape)

(1, 8)


## Step 6. Display the head to check if working

In [15]:
df.head()

,I_COMMODITY,CTY_CODE,CNT_CHA_MO,CNT_WGT_MO,CNT_VAL_MO,GEN_VAL_MO,time,COMM_LVL
0,8517,5700,3532735,5406638,221291238,7185905349,2023-10,HS4


In [17]:
for level in ["HS4", "HS6"]:
    p = {
        "get": "I_COMMODITY,CNT_CHA_MO,CNT_WGT_MO",
        "time": "2023-10",
        "COMM_LVL": level,
        "CTY_CODE": "5700",
        "I_COMMODITY": "*",
        "key": API_KEY,
    }
    u = BASE_URL + "?" + urllib.parse.urlencode(p, safe=",*")
    with urllib.request.urlopen(u, timeout=120) as response:
        r = json.loads(response.read().decode())
    d = pd.DataFrame(r[1:], columns=r[0])
    d = d.loc[:, ~d.columns.duplicated()]
    w = pd.to_numeric(d["CNT_WGT_MO"], errors="coerce")
    print(level, "| product codes:", len(d), "| with container trade:", (w > 0).sum(),
          "| share empty:", round(1 - (w > 0).mean(), 2))

HS4 | product codes: 1142 | with container trade: 982 | share empty: 0.14
HS6 | product codes: 4606 | with container trade: 3589 | share empty: 0.22


In [ ]:
China for Oct 2023 at the national level: 14% of HS4 codes and 22% of HS6 codes -- no container trade.
HS6- cells can get much sparser once I split by district and add smaller countries.
    

## Step 7. More 

In [13]:
params = {
    "get": "I_COMMODITY,CTY_CODE,DISTRICT,CNT_CHA_MO,CNT_WGT_MO,CNT_VAL_MO,GEN_VAL_MO",
    "time": "2023-10",
    "COMM_LVL": "HS4",
    "CTY_CODE": "5700",
    "I_COMMODITY": "*",
    "DISTRICT": "*",
    "key": API_KEY,
}
url = BASE_URL + "?" + urllib.parse.urlencode(params, safe=",*")
with urllib.request.urlopen(url, timeout=120) as response: #REMOVE TIMEOUT AND TRYYYYYY
    rows = json.loads(response.read().decode())
one_month = pd.DataFrame(rows[1:], columns=rows[0])
print(one_month.shape)
print("District codes:", sorted(one_month["DISTRICT"].unique()))
one_month.head()

TimeoutError: The read operation timed out

## Column guide
* `I_COMMODITY`: HS4 product code
* `CTY_CODE`: origin country code (5700 = China)
* `DISTRICT`: U.S. customs district code
* `CNT_CHA_MO`: containerized vessel charges, USD
* `CNT_WGT_MO`: containerized vessel shipping weight, kg
* `CNT_VAL_MO`: containerized vessel value, USD
* `GEN_VAL_MO`: total general import value, USD